# Smoke test fine-tune nhận diện bệnh cà phê
Chạy từng ô theo thứ tự. Trên Colab: `Runtime > Change runtime type > T4 GPU` trước khi chạy.
**Chỉ kiểm tra pipeline, không đại diện cho Qwen3.8-27B.** Chỉ dùng dữ liệu giả hoặc dữ liệu công khai ở đây, không đưa ảnh khách hàng lên cloud.

## 0. Đưa mã lên (Colab) và kiểm GPU
Ô dưới mở hộp chọn tệp: chọn `nextfarm-pest-finetune-colab.zip`. Nếu hộp chọn tệp không hiện sau ~20 giây, bấm dừng ô rồi chạy lại. Nếu đã có thư mục `src/` thì ô này bỏ qua bước tải.

In [ ]:
import os, sys, subprocess, zipfile
os.environ["PYTHONUTF8"] = "1"
if not os.path.isdir("src"):
    from google.colab import files  # chỉ có trên Colab
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall(".")
print("Python", sys.version.split()[0], "| thư mục:", os.getcwd(), "| có src:", os.path.isdir("src"))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "KHÔNG THẤY GPU: bật T4 trong Runtime > Change runtime type")

In [ ]:
# DATASET: "rocole" = ảnh lá cà phê thật (tải ~141 ảnh từ Mendeley); "synthetic" = ảnh giả.
DATASET = "rocole"
# Model cho T4 (16 GB). Trên máy 4 GB VRAM đổi thành "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit".
MODEL = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
STEPS = 60 if DATASET == "rocole" else 30   # 99 ảnh train, batch hiệu dụng 4: 25 step = 1 epoch
# Cạnh dài ảnh đưa vào model. Ảnh RoCoLe gốc 2048x1152; ở 336 thì vết nhện đỏ gần như không còn thấy.
IMG = 672 if DATASET == "rocole" else 336

## 1. Dữ liệu
Với `rocole`: tải tệp nhãn, dựng manifest, tách theo cây, rồi chỉ tải các ảnh đã chọn (vài phút). Luôn tạo thêm 20 ảnh ngoài lề (ảnh rác giả).
Dòng cuối phải là `OK: 0 lỗi.` Xem thêm `data/splits/summary.json` và `data/raw/rocole/SOURCE.txt` (nguồn + giấy phép).

In [ ]:
%pip install -q pillow openpyxl
if DATASET == "rocole":
    !python src/make_synthetic.py --ood-only
    !python src/fetch_rocole.py --force
    !python src/prepare_dataset.py
    !python src/fetch_rocole.py --images
else:
    !python src/make_synthetic.py --force
    !python src/prepare_dataset.py
!python src/validate_dataset.py

In [ ]:
# Nhìn tận mắt dữ liệu: 2 ảnh train mỗi nhãn, ở đúng kích thước model sẽ thấy (IMG).
import json
from PIL import Image
from IPython.display import display
rows = [json.loads(l) for l in open("data/splits/train.jsonl", encoding="utf-8")]
for lab in sorted({r["label"] for r in rows}):
    for r in [r for r in rows if r["label"] == lab][:2]:
        im = Image.open("data/raw/" + r["image_path"]); im.thumbnail((IMG, IMG))
        print(r["sample_id"], "| cây:", r["group_id"], "| target:", r["target"]); display(im)

## 2. Cài thư viện train (chỉ chạy một lần mỗi phiên, vài phút)
Nếu ô này báo cần khởi động lại runtime: `Runtime > Restart session`, rồi chạy lại ô 0 và ô đặt `DATASET`/`MODEL` (không cần chạy lại ô 1).

In [ ]:
%pip install -q unsloth
!mkdir -p outputs/meta && pip freeze > outputs/meta/pip-freeze.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 3. Kiểm khả thi nhanh (3 step, 3 ảnh)
Mục đích: lộ lỗi phiên bản thư viện trong vài phút, trước khi chạy đủ. Lần này có một đoạn mã MỚI chưa từng chạy: tính loss chỉ trên câu trả lời.
Nếu lỗi: sao chép NGUYÊN traceback gửi lại. Để đi tiếp tạm thời: thêm `--loss-on-all-tokens` (nếu lỗi ở collator) hoặc `--eval-steps 0` (nếu lỗi ở bước đo val) vào các lệnh ở ô 3–6.

In [ ]:
!python src/train_eval.py --model {MODEL} --img-size {IMG} --skip-baseline --max-steps 3 --limit-test 3

## 4. Chạy đủ: B0, B1 -> train -> FT, FT nạp lại, FT với prompt đang chạy thật, ảnh ngoài lề
Model gốc giờ được viết tới 512 token nên B0/B1 chậm hơn lần trước (ước chừng 10–20 giây mỗi ảnh).

In [ ]:
!python src/train_eval.py --model {MODEL} --img-size {IMG} --max-steps {STEPS}

## 5. Đối chứng A: mã nhãn vô nghĩa (class_a/b/c), nhãn đúng ảnh
Model gốc không thể đoán `class_b` nghĩa là gì, nên accuracy ở đây là phần model THẬT SỰ học được từ ảnh + nhãn train.

In [ ]:
!python src/train_eval.py --model {MODEL} --img-size {IMG} --max-steps {STEPS} --opaque-labels

## 6. Đối chứng B: mã nhãn vô nghĩa VÀ xáo trộn
Nhãn train không còn liên quan tới ảnh. Accuracy phải rơi về khoảng 33%. Nếu vẫn cao thì pipeline có rò rỉ hoặc chấm sai.

In [ ]:
!python src/train_eval.py --model {MODEL} --img-size {IMG} --max-steps {STEPS} --shuffle-labels

## 7. So sánh và kiểm chéo

In [ ]:
!python src/compare.py

## 8. Tải kết quả về máy
Gói `outputs/` và `data/splits/` (bỏ adapter, checkpoint, ảnh) thành `outputs_<DATASET>.zip`. Trên máy: chuột phải > Extract All vào thư mục `outputs\` của dự án, kết quả nằm ở `outputs\outputs_<DATASET>\`.

In [ ]:
import os, zipfile
name = f"outputs_{DATASET}.zip"
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for d, dirs, fs in os.walk("outputs"):
        dirs[:] = [x for x in dirs if not x.startswith(("adapter", "trainer"))]
        for f in fs:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, "outputs"))
    for f in sorted(os.listdir("data/splits")):
        z.write(os.path.join("data/splits", f), "splits/" + f)
    for f in ("data/ood.jsonl", "data/raw/rocole/SOURCE.txt"):
        if os.path.exists(f):
            z.write(f, "splits/" + os.path.basename(f))
print("Đã gói", name, os.path.getsize(name) // 1024, "KB")
try:
    from google.colab import files
    files.download(name)
except ImportError:
    print("Không phải Colab: lấy tệp zip ở thư mục làm việc")